---
## 1. Carregar Datasets

In [ ]:
# Importar bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import re
import warnings

warnings.filterwarnings('ignore')

# Configurar estilo de visualizações
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("✓ Bibliotecas importadas com sucesso")

In [ ]:
# Definir caminho dos datasets
datasets_path = Path("datasets")

# Carregar datasets
df_clientes = pd.read_csv(datasets_path / "clientes.csv")
df_produtos = pd.read_csv(datasets_path / "produtos.csv")
df_vendas = pd.read_csv(datasets_path / "vendas.csv")

# Armazenar em dicionário para facilitar iteração
datasets = {
    "clientes": df_clientes,
    "produtos": df_produtos,
    "vendas": df_vendas
}

# Exibir informações dos datasets
print("\n" + "="*80)
print("DATASETS CARREGADOS")
print("="*80)

for nome, df in datasets.items():
    print(f"\n📊 {nome.upper()}")
    print(f"   Linhas: {len(df)} | Colunas: {len(df.columns)}")
    print(f"   Memória: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
    print(f"   Colunas: {', '.join(df.columns.tolist())}")

---
## 2. Identificar Problemas de Qualidade

In [ ]:
# Função para detectar problemas de qualidade
def detectar_valores_ausentes(df, nome_dataset):
    """Detecta valores ausentes (NaN) em todas as colunas"""
    problemas = []
    
    for coluna in df.columns:
        nulos = df[coluna].isna().sum()
        if nulos > 0:
            problemas.append({
                "dataset": nome_dataset,
                "coluna": coluna,
                "tipo_problema": "Valores Ausentes",
                "dimensao": "Completude",
                "quantidade_afetada": nulos,
                "percentual_afetado": (nulos / len(df)) * 100
            })
    
    return problemas

def detectar_duplicatas(df, nome_dataset):
    """Detecta duplicatas completas e por chave primária"""
    problemas = []
    
    # Duplicatas completas
    duplicatas = df.duplicated().sum()
    if duplicatas > 0:
        problemas.append({
            "dataset": nome_dataset,
            "coluna": "[Registro Completo]",
            "tipo_problema": "Registros Duplicados",
            "dimensao": "Unicidade",
            "quantidade_afetada": duplicatas,
            "percentual_afetado": (duplicatas / len(df)) * 100
        })
    
    # Duplicatas por coluna (potencial chave primária)
    if "id_" in nome_dataset or nome_dataset == "clientes":
        coluna_id = f"id_{nome_dataset.rstrip('s')}"
        if coluna_id in df.columns:
            duplicatas_id = df[coluna_id].duplicated().sum()
            if duplicatas_id > 0:
                problemas.append({
                    "dataset": nome_dataset,
                    "coluna": coluna_id,
                    "tipo_problema": "IDs Duplicados",
                    "dimensao": "Unicidade",
                    "quantidade_afetada": duplicatas_id,
                    "percentual_afetado": (duplicatas_id / len(df)) * 100
                })
    
    return problemas

def detectar_tipo_dados_incorretos(df, nome_dataset):
    """Detecta tipos de dados potencialmente incorretos"""
    problemas = []
    
    # Verificar datas que deviam ser datetime
    colunas_data = [col for col in df.columns if 'data' in col.lower()]
    for coluna in colunas_data:
        if df[coluna].dtype == 'object':
            try:
                pd.to_datetime(df[coluna], errors='coerce')
                invalidos = pd.to_datetime(df[coluna], errors='coerce').isna().sum()
                if invalidos > 0:
                    problemas.append({
                        "dataset": nome_dataset,
                        "coluna": coluna,
                        "tipo_problema": "Formato de Data Inválido",
                        "dimensao": "Validade",
                        "quantidade_afetada": invalidos,
                        "percentual_afetado": (invalidos / len(df)) * 100
                    })
            except:
                pass
    
    # Verificar números que deviam ser numéricos
    colunas_numericas = [col for col in df.columns if 'preco' in col.lower() or 'valor' in col.lower() or 'quantidade' in col.lower() or 'estoque' in col.lower()]
    for coluna in colunas_numericas:
        if coluna in df.columns and df[coluna].dtype == 'object':
            problemas.append({
                "dataset": nome_dataset,
                "coluna": coluna,
                "tipo_problema": "Tipo Numérico Incorreto",
                "dimensao": "Validade",
                "quantidade_afetada": len(df),
                "percentual_afetado": 100.0
            })
    
    return problemas

def detectar_outliers_numericos(df, nome_dataset):
    """Detecta outliers usando IQR (Interquartile Range)"""
    problemas = []
    
    for coluna in df.select_dtypes(include=[np.number]).columns:
        Q1 = df[coluna].quantile(0.25)
        Q3 = df[coluna].quantile(0.75)
        IQR = Q3 - Q1
        
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        
        outliers = ((df[coluna] < limite_inferior) | (df[coluna] > limite_superior)).sum()
        
        if outliers > 0 and outliers / len(df) <= 0.1:  # Apenas se < 10%
            problemas.append({
                "dataset": nome_dataset,
                "coluna": coluna,
                "tipo_problema": "Possíveis Outliers",
                "dimensao": "Acurácia",
                "quantidade_afetada": outliers,
                "percentual_afetado": (outliers / len(df)) * 100
            })
    
    return problemas

def detectar_validacoes_pattern(df, nome_dataset):
    """Detecta valores que não correspondem a padrões esperados"""
    problemas = []
    
    # Validação de email
    if 'email' in df.columns:
        email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
        emails_invalidos = df['email'].apply(
            lambda x: not re.match(email_pattern, str(x)) if pd.notna(x) else False
        ).sum()
        
        if emails_invalidos > 0:
            problemas.append({
                "dataset": nome_dataset,
                "coluna": "email",
                "tipo_problema": "Email Inválido",
                "dimensao": "Validade",
                "quantidade_afetada": emails_invalidos,
                "percentual_afetado": (emails_invalidos / len(df)) * 100
            })
    
    # Validação de telefone (11 dígitos)
    if 'telefone' in df.columns:
        telefone_pattern = r'^\d{11}$'
        telefones_invalidos = df['telefone'].apply(
            lambda x: not re.match(telefone_pattern, str(x)) if pd.notna(x) else False
        ).sum()
        
        if telefones_invalidos > 0:
            problemas.append({
                "dataset": nome_dataset,
                "coluna": "telefone",
                "tipo_problema": "Telefone Inválido",
                "dimensao": "Validade",
                "quantidade_afetada": telefones_invalidos,
                "percentual_afetado": (telefones_invalidos / len(df)) * 100
            })
    
    # Validação de estado (2 caracteres, maiúsculas)
    if 'estado' in df.columns:
        estado_pattern = r'^[A-Z]{2}$'
        estados_invalidos = df['estado'].apply(
            lambda x: not re.match(estado_pattern, str(x)) if pd.notna(x) else False
        ).sum()
        
        if estados_invalidos > 0:
            problemas.append({
                "dataset": nome_dataset,
                "coluna": "estado",
                "tipo_problema": "Estado Inválido",
                "dimensao": "Consistência",
                "quantidade_afetada": estados_invalidos,
                "percentual_afetado": (estados_invalidos / len(df)) * 100
            })
    
    return problemas

def detectar_valores_invalidos_intervalo(df, nome_dataset):
    """Detecta valores fora do intervalo esperado"""
    problemas = []
    
    # Preço deve ser > 0
    if 'preco' in df.columns:
        preco_invalido = (df['preco'] <= 0).sum()
        if preco_invalido > 0:
            problemas.append({
                "dataset": nome_dataset,
                "coluna": "preco",
                "tipo_problema": "Preço Inválido (<= 0)",
                "dimensao": "Validade",
                "quantidade_afetada": preco_invalido,
                "percentual_afetado": (preco_invalido / len(df)) * 100
            })
    
    # Quantidade deve ser > 0
    if 'quantidade' in df.columns:
        quantidade_invalida = (df['quantidade'] <= 0).sum()
        if quantidade_invalida > 0:
            problemas.append({
                "dataset": nome_dataset,
                "coluna": "quantidade",
                "tipo_problema": "Quantidade Inválida (<= 0)",
                "dimensao": "Validade",
                "quantidade_afetada": quantidade_invalida,
                "percentual_afetado": (quantidade_invalida / len(df)) * 100
            })
    
    # Estoque deve ser >= 0
    if 'estoque' in df.columns:
        estoque_invalido = (df['estoque'] < 0).sum()
        if estoque_invalido > 0:
            problemas.append({
                "dataset": nome_dataset,
                "coluna": "estoque",
                "tipo_problema": "Estoque Negativo",
                "dimensao": "Validade",
                "quantidade_afetada": estoque_invalido,
                "percentual_afetado": (estoque_invalido / len(df)) * 100
            })
    
    return problemas

# Executar detecção de todos os problemas
print("\n" + "="*80)
print("IDENTIFICANDO PROBLEMAS DE QUALIDADE")
print("="*80)

todos_os_problemas = []

for nome, df in datasets.items():
    print(f"\n🔍 Analisando: {nome.upper()}")
    
    problemas_ausentes = detectar_valores_ausentes(df, nome)
    problemas_duplicatas = detectar_duplicatas(df, nome)
    problemas_tipos = detectar_tipo_dados_incorretos(df, nome)
    problemas_outliers = detectar_outliers_numericos(df, nome)
    problemas_pattern = detectar_validacoes_pattern(df, nome)
    problemas_intervalo = detectar_valores_invalidos_intervalo(df, nome)
    
    problemas_dataset = (problemas_ausentes + problemas_duplicatas + problemas_tipos + 
                        problemas_outliers + problemas_pattern + problemas_intervalo)
    
    todos_os_problemas.extend(problemas_dataset)
    print(f"   ✓ {len(problemas_dataset)} problemas encontrados")

# Criar DataFrame com todos os problemas
df_problemas = pd.DataFrame(todos_os_problemas)

print(f"\n✓ Total de problemas identificados: {len(df_problemas)}")

---
## 3. Classificar Problemas por Dimensão

In [ ]:
# Agrupar por dimensão de qualidade
print("\n" + "="*80)
print("CLASSIFICAÇÃO POR DIMENSÃO DE QUALIDADE")
print("="*80)

dimensoes_resumo = df_problemas.groupby('dimensao').agg({
    'tipo_problema': 'count',
    'percentual_afetado': 'sum'
}).rename(columns={'tipo_problema': 'quantidade_problemas'})

print("\n📊 Resumo por Dimensão:")
print(dimensoes_resumo.to_string())

# Tabela detalhada por dimensão
print("\n\n📋 Detalhes por Dimensão:")

for dimensao in df_problemas['dimensao'].unique():
    problemas_dim = df_problemas[df_problemas['dimensao'] == dimensao]
    print(f"\n{'─'*80}")
    print(f"🔹 {dimensao.upper()}")
    print(f"{'─'*80}")
    print(f"Total de problemas: {len(problemas_dim)}")
    print(f"Percentual total afetado: {problemas_dim['percentual_afetado'].sum():.2f}%")
    print(f"\nProblemas:")
    for _, problema in problemas_dim.iterrows():
        print(f"  • {problema['tipo_problema']} ({problema['dataset']}.{problema['coluna']})")
        print(f"    → {problema['quantidade_afetada']} registros | {problema['percentual_afetado']:.2f}% afetados")

In [ ]:
# Visualizar distribuição por dimensão
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Quantidade de problemas por dimensão
dimensoes_count = df_problemas['dimensao'].value_counts()
colors = sns.color_palette("husl", len(dimensoes_count))
axes[0].bar(dimensoes_count.index, dimensoes_count.values, color=colors)
axes[0].set_title("Quantidade de Problemas por Dimensão", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Quantidade de Problemas")
axes[0].set_xlabel("Dimensão de Qualidade")
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(dimensoes_count.values):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# Gráfico 2: Percentual afetado por dimensão
percentuais_dim = df_problemas.groupby('dimensao')['percentual_afetado'].sum().sort_values(ascending=False)
axes[1].bar(percentuais_dim.index, percentuais_dim.values, color=colors)
axes[1].set_title("Percentual Total Afetado por Dimensão", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Percentual Afetado (%)")
axes[1].set_xlabel("Dimensão de Qualidade")
axes[1].tick_params(axis='x', rotation=45)
for i, (idx, v) in enumerate(percentuais_dim.items()):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualização de dimensões concluída")

---
## 4. Calcular Impacto dos Problemas

In [ ]:
# Função para calcular severidade
def calcular_severidade(percentual):
    """Classifica severidade baseada no percentual afetado"""
    if percentual >= 50:
        return "CRÍTICA"
    elif percentual >= 20:
        return "ALTA"
    elif percentual >= 10:
        return "MÉDIA"
    else:
        return "BAIXA"

# Calcular impacto detalhado
df_problemas['severidade'] = df_problemas['percentual_afetado'].apply(calcular_severidade)

# Calcular score de impacto (0-100)
df_problemas['score_impacto'] = (
    (df_problemas['percentual_afetado'] / 100) * 100
)

print("\n" + "="*80)
print("ANÁLISE DE IMPACTO DOS PROBLEMAS")
print("="*80)

# Tabela de impacto
df_impacto = df_problemas[[
    'dataset', 'coluna', 'tipo_problema', 'dimensao', 
    'quantidade_afetada', 'percentual_afetado', 'severidade', 'score_impacto'
]].copy()

df_impacto = df_impacto.sort_values('score_impacto', ascending=False)

print("\n📊 Tabela de Impacto (Top 15 Problemas):")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print(df_impacto.head(15).to_string(index=False))

# Resumo por severidade
print("\n\n📈 Resumo por Severidade:")
severidade_resumo = df_problemas['severidade'].value_counts().sort_index(key=lambda x: x.map({'CRÍTICA': 0, 'ALTA': 1, 'MÉDIA': 2, 'BAIXA': 3}))
print(severidade_resumo.to_string())

print(f"\n✓ Score de impacto calculado para todos os problemas")

In [ ]:
# Visualizar impacto por problema
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top 10 problemas por impacto
top_10 = df_impacto.head(10)
cores_severidade = {'CRÍTICA': '#d62728', 'ALTA': '#ff7f0e', 'MÉDIA': '#2ca02c', 'BAIXA': '#1f77b4'}
cores = [cores_severidade.get(s, '#1f77b4') for s in top_10['severidade']]

problemas_labels = [f"{row['tipo_problema']}\n({row['dataset']}.{row['coluna']})" 
                     for _, row in top_10.iterrows()]

axes[0].barh(range(len(top_10)), top_10['percentual_afetado'].values, color=cores)
axes[0].set_yticks(range(len(top_10)))
axes[0].set_yticklabels(problemas_labels, fontsize=9)
axes[0].set_xlabel("Percentual de Registros Afetados (%)")
axes[0].set_title("Top 10 Problemas por Impacto", fontsize=12, fontweight='bold')
axes[0].invert_yaxis()
for i, v in enumerate(top_10['percentual_afetado'].values):
    axes[0].text(v + 1, i, f"{v:.1f}%", va='center', fontweight='bold')

# Distribuição de severidade
severidade_order = ['CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA']
severidade_counts = []
for sev in severidade_order:
    count = (df_problemas['severidade'] == sev).sum()
    severidade_counts.append(count)

cores_sev = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']
wedges, texts, autotexts = axes[1].pie(severidade_counts, labels=severidade_order, autopct='%1.1f%%',
                                         colors=cores_sev, startangle=90)
axes[1].set_title("Distribuição de Problemas por Severidade", fontsize=12, fontweight='bold')
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.show()

print("✓ Visualização de impacto concluída")

---
## 5. Priorizar por Criticidade

In [ ]:
# Função para calcular score de prioridade
def calcular_score_prioridade(row):
    
""Calcula score de prioridade (0-100)"""
    # Fatores:
    # 1. Percentual afetado (peso: 50%)
    # 2. Tipo de dimensão (peso: 30%)
    # 3. Frequência em dataset (peso: 20%)
    
    peso_percentual = row['percentual_afetado']
    
    # Dimensões críticas têm maior peso
    peso_dimensao = {
        'Completude': 100,
        'Unicidade': 90,
        'Validade': 80,
        'Consistência': 70,
        'Pontualidade': 60,
        'Acurácia': 50
    }
    peso_dim = peso_dimensao.get(row['dimensao'], 50)
    
    # Score final: média ponderada
    score = (peso_percentual * 0.5) + (peso_dim * 0.3) + (row['percentual_afetado'] * 0.2)
    
    return min(score, 100)  # Cap em 100

df_problemas['score_prioridade'] = df_problemas.apply(calcular_score_prioridade, axis=1)

# Criar ranking
df_ranking = df_problemas[[
    'dataset', 'coluna', 'tipo_problema', 'dimensao',
    'quantidade_afetada', 'percentual_afetado', 'severidade', 'score_prioridade'
]].copy()

df_ranking = df_ranking.sort_values('score_prioridade', ascending=False).reset_index(drop=True)
df_ranking['ranking'] = range(1, len(df_ranking) + 1)

print("\n" + "="*100)
print("RANKING DE PRIORIDADE DOS PROBLEMAS")
print("="*100)

# Exibir ranking completo
pd.set_option('display.max_rows', None)
display_df = df_ranking[['ranking', 'dataset', 'coluna', 'tipo_problema', 'dimensao', 
                          'percentual_afetado', 'severidade', 'score_prioridade']].copy()

print("\n📋 Ranking Completo:")
print(display_df.to_string(index=False))

In [ ]:
# Criar resumo executivo por dataset
print("\n\n" + "="*100)
print("RESUMO EXECUTIVO POR DATASET")
print("="*100)

for dataset_nome in datasets.keys():
    problemas_dataset = df_ranking[df_ranking['dataset'] == dataset_nome]
    
    if len(problemas_dataset) > 0:
        print(f"\n{'─'*100}")
        print(f"📊 {dataset_nome.upper()} - {len(datasets[dataset_nome])} registros")
        print(f"{'─'*100}")
        print(f"Total de problemas: {len(problemas_dataset)}")
        print(f"Problemas críticos: {len(problemas_dataset[problemas_dataset['severidade'] == 'CRÍTICA'])}")
        print(f"Problemas de alta severidade: {len(problemas_dataset[problemas_dataset['severidade'] == 'ALTA'])}")
        
        print(f"\nTop 5 Prioridades para {dataset_nome}:")
        top_5 = problemas_dataset.head(5)
        for idx, (_, row) in enumerate(top_5.iterrows(), 1):
            print(f"  {idx}. [{row['severidade']}] {row['tipo_problema']} ({row['coluna']})")
            print(f"     → {row['percentual_afetado']:.1f}% dos registros | Score: {row['score_prioridade']:.1f}")

In [ ]:
# Visualizar ranking de prioridade
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Gráfico de Prioridade - Top 12
top_12 = df_ranking.head(12)
cores_sev = {'CRÍTICA': '#d62728', 'ALTA': '#ff7f0e', 'MÉDIA': '#2ca02c', 'BAIXA': '#1f77b4'}
cores = [cores_sev.get(s, '#1f77b4') for s in top_12['severidade']]

problemas_str = [f"#{int(r)} {t}\n({d})" 
                  for r, t, d in zip(top_12['ranking'], top_12['tipo_problema'], top_12['dataset'])]

axes[0, 0].barh(range(len(top_12)), top_12['score_prioridade'].values, color=cores)
axes[0, 0].set_yticks(range(len(top_12)))
axes[0, 0].set_yticklabels(problemas_str, fontsize=8)
axes[0, 0].set_xlabel("Score de Prioridade")
axes[0, 0].set_title("Top 12 Problemas por Prioridade", fontsize=12, fontweight='bold')
axes[0, 0].invert_yaxis()
for i, v in enumerate(top_12['score_prioridade'].values):
    axes[0, 0].text(v + 1, i, f"{v:.1f}", va='center', fontsize=8, fontweight='bold')

# 2. Matriz de Impacto x Dimensão
matriz = df_ranking.pivot_table(
    values='score_prioridade',
    index='dimensao',
    columns='severidade',
    aggfunc='count',
    fill_value=0
)
matriz = matriz[['CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA']]

sns.heatmap(matriz, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0, 1], cbar_kws={'label': 'Quantidade'})
axes[0, 1].set_title("Matriz: Dimensão vs Severidade", fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel("Dimensão")
axes[0, 1].set_xlabel("Severidade")

# 3. Distribuição de Score por Dataset
for dataset_nome in datasets.keys():
    data = df_ranking[df_ranking['dataset'] == dataset_nome]['score_prioridade']
    axes[1, 0].hist(data, alpha=0.6, label=dataset_nome, bins=10)

axes[1, 0].set_xlabel("Score de Prioridade")
axes[1, 0].set_ylabel("Frequência")
axes[1, 0].set_title("Distribuição de Scores por Dataset", fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Timeline de Ação (Recomendação)
score_ranges = ['0-20 (Baixa)', '20-40 (Média)', '40-60 (Alta)', '60-80 (Crítica)', '80-100 (Crítica Alto)']
counts = [
    len(df_ranking[(df_ranking['score_prioridade'] >= 0) & (df_ranking['score_prioridade'] < 20)]),
    len(df_ranking[(df_ranking['score_prioridade'] >= 20) & (df_ranking['score_prioridade'] < 40)]),
    len(df_ranking[(df_ranking['score_prioridade'] >= 40) & (df_ranking['score_prioridade'] < 60)]),
    len(df_ranking[(df_ranking['score_prioridade'] >= 60) & (df_ranking['score_prioridade'] < 80)]),
    len(df_ranking[(df_ranking['score_prioridade'] >= 80)])
]

colors_timeline = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#8b0000']
axes[1, 1].bar(range(len(score_ranges)), counts, color=colors_timeline)
axes[1, 1].set_xticks(range(len(score_ranges)))
axes[1, 1].set_xticklabels(score_ranges, rotation=45, ha='right')
axes[1, 1].set_ylabel("Quantidade de Problemas")
axes[1, 1].set_title("Distribuição por Faixa de Prioridade", fontsize=12, fontweight='bold')
for i, v in enumerate(counts):
    axes[1, 1].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualização de priorização concluída")

In [ ]:
# Recomendações de ação
print("\n" + "="*100)
print("RECOMENDAÇÕES DE AÇÃO")
print("="*100)

# Problemas críticos que precisam ação imediata
criticos = df_ranking[df_ranking['severidade'] == 'CRÍTICA']
if len(criticos) > 0:
    print("\n🔴 AÇÃO IMEDIATA (Problemas Críticos):")
    print(f"    Total: {len(criticos)} problemas")
    print(f"    Prazo: HOJE")
    for idx, row in criticos.head(5).iterrows():
        print(f"    • {row['tipo_problema']} em {row['dataset']}.{row['coluna']} ({row['percentual_afetado']:.1f}%)")

# Problemas de alta prioridade
altos = df_ranking[df_ranking['severidade'] == 'ALTA']
if len(altos) > 0:
    print("\n🟠 ALTA PRIORIDADE:")
    print(f"    Total: {len(altos)} problemas")
    print(f"    Prazo: Esta semana")
    for idx, row in altos.head(3).iterrows():
        print(f"    • {row['tipo_problema']} em {row['dataset']}.{row['coluna']} ({row['percentual_afetado']:.1f}%)")

# Problemas de média prioridade
medios = df_ranking[df_ranking['severidade'] == 'MÉDIA']
if len(medios) > 0:
    print("\n🟡 MÉDIA PRIORIDADE:")
    print(f"    Total: {len(medios)} problemas")
    print(f"    Prazo: Este mês")

# Problemas de baixa prioridade
baixos = df_ranking[df_ranking['severidade'] == 'BAIXA']
if len(baixos) > 0:
    print("\n🟢 BAIXA PRIORIDADE:")
    print(f"    Total: {len(baixos)} problemas")
    print(f"    Prazo: Próximos 3 meses")

In [ ]:
# Salvar relatório em CSV para referência
print("\n" + "="*100)
print("EXPORTANDO RELATÓRIOS")
print("="*100)

# Salvar ranking completo
df_ranking.to_csv('relatorio_problemas_qualidade.csv', index=False, encoding='utf-8')
print("\n✓ Relatório salvo: relatorio_problemas_qualidade.csv")

# Estatísticas finais
print("\n" + "="*100)
print("ESTATÍSTICAS FINAIS")
print("="*100)

print(f"""
📊 RESUMO EXECUTIVO
{'─'*100}
Total de problemas identificados:     {len(df_ranking)}
Total de datasets analisados:         {len(datasets)}
Total de registros analisados:        {sum(len(df) for df in datasets.values())}

Distribuição por Severidade:
  • Crítica:  {len(criticos):2d} problemas ({len(criticos)/len(df_ranking)*100:.1f}%)
  • Alta:     {len(altos):2d} problemas ({len(altos)/len(df_ranking)*100:.1f}%)
  • Média:    {len(medios):2d} problemas ({len(medios)/len(df_ranking)*100:.1f}%)
  • Baixa:    {len(baixos):2d} problemas ({len(baixos)/len(df_ranking)*100:.1f}%)

Distribuição por Dimensão:
  • Completude:     {len(df_ranking[df_ranking['dimensao']=='Completude']):2d} problemas
  • Unicidade:      {len(df_ranking[df_ranking['dimensao']=='Unicidade']):2d} problemas
  • Validade:       {len(df_ranking[df_ranking['dimensao']=='Validade']):2d} problemas
  • Consistência:   {len(df_ranking[df_ranking['dimensao']=='Consistência']):2d} problemas
  • Acurácia:       {len(df_ranking[df_ranking['dimensao']=='Acurácia']):2d} problemas

Top 3 Datasets com Problemas:
""")

dataset_counts = df_ranking['dataset'].value_counts().head(3)
for dataset, count in dataset_counts.items():
    print(f"  • {dataset:12s}: {count:2d} problemas")

print(f"""
{'─'*100}
Gerado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
""")